# Verification of the modulo-49 identities in Sage

Following the modulo-256 and modulo-81 playgrounds, we define the relations, load the precomputed source modules and recorded intermediate elements, and check their defining equations.

Here the saved data use **recursive complements**. We also check the transfer maps, their compatibility with the two Hecke operators, and generation by transferred elements together with the complement. A complement check alone is not a whole-source verification.

The Manin presentations and Hecke matrices are inputs. This notebook checks the finite-source conditions; all-weight propagation and passage to eigenvalues are the separate arguments in the manuscript. No search for division outputs is performed.


In [1]:
import gzip
import hashlib
import json
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np
from sage.all import *

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    raise RuntimeError('Open this notebook from the repository directory.')
sys.path.insert(0, str(ROOT / 'python'))
from verify_hecke_relations import relation_spec
from p7_mod49 import p7_mod49_relation_polynomials

p = 7
relation_period = 42
VERIFICATION_DIRECTORY = ROOT / 'verification_data/mod49_compact'


## 1. Define all the relations

Put $j=(d+2q)\bmod6$, $r=(d+14q)\bmod42$, and $t_n=n^{q7^{m-1}}T_n$ on the signed source with sign $(-1)^q$. The twist is applied exactly once.

We use $\mathcal Q_j=F_j(t_3)/49$ and $\mathcal G_j=(t_{29}-2-t_3(t_3-\alpha_j)(t_3-\beta_j))/7$, interpreted as division relations. Their terminal polynomial is $H(X)=(X^7-X)^3$.

The larger selector polynomials are loaded as explicit Sage polynomials from the repository's coefficient table. In that table the two coordinates $X,Y$ stand for $\mathcal Q_j,\mathcal G_j$. The additional coordinate $U$ or $V$ stands for the corresponding projected digit. The polynomials are cubed over $\mathbf F_7$ **before** their coefficients are lifted.

For each centre $c$, the input is multiplied by $E_{j,c}(t_3)^6$. The projected digits have numerators $E_{j,c}(t_3)^6(t_3-c)$ and $E_{j,c}(t_3)^6(t_{29}-2)$ and divisor $7$.


In [2]:
S.<X> = PolynomialRing(ZZ)
F_j = {
    0: X^6+7*X^5+45*X^4+7*X^3+4*X^2+7*X,
    2: X^6+42*X^5+6*X^4+9*X^2+14*X,
    4: X^6+7*X^5+47*X^4+14*X^3+X^2+28*X,
}
centres = {0: (0,3,4), 2: (0,2,5), 4: (0,1,6)}
H = (X^7-X)^3
selector_data = p7_mod49_relation_polynomials()
R_rc = selector_data['R']       # R_rc[r,c,nu] = R_{r,c}^{(nu)}(X,Y).
phi_rc = selector_data['phi']   # phi_rc[r,c] = phi_{r,c}(X,Y).
psi_rc = selector_data['psi']   # psi_rc[r,c] = psi_{r,c}(X,Y).
assert selector_data['fixed_relation_power'] == 3
# Each record contains its residue, centre, label and explicit Sage polynomial.
selector_polynomials = selector_data['relations']

def lagrange_lift(j, c):
    """
    Return the coefficientwise lift of the Lagrange selector E_{j,c}.
    First compute over F_7; then lift each coefficient to 0,...,6.
    This is a choice of polynomial lift, not a ring map F_7 -> Z/343Z.
    """
    P = PolynomialRing(GF(7), 'x')
    x = P.gen()
    e = prod((x-other)/GF(7)(c-other) for other in centres[j] if other != c)
    return S([ZZ(coefficient) for coefficient in e.list()])

def make_specification(r, m):
    """
    Encode the displayed relations in the format of the recorded data.
    At precision 49 test H(G); at precision 343 test H(Q) and every
    selector polynomial for residue r, including the projected digits.
    Return the complete presentation, with its prescribed input selectors.
    """
    j = r % 6
    alpha, beta = centres[j][1:]
    names = ('T3','T29','G') if m == 2 else (
        'T3','T29','Q','G','JU0','JV0','JU1','JV1','JU2','JV2')
    P = PolynomialRing(Integers(7^m), names=names)
    v = dict(zip(names, P.gens()))
    T3, T29, G = v['T3'], v['T29'], v['G']
    divisions = {'G': (T29-2-T3*(T3-alpha)*(T3-beta), 1)}
    if m == 2:
        return relation_spec([('G', (G^7-G)^3, 1)],
                             {'T3':3,'T29':29}, divisions)
    Q = v['Q']
    divisions['Q'] = (F_j[j](T3), 2)
    relations = [('Q', (Q^7-Q)^3, 1)]
    inputs = []
    for i,c in enumerate(centres[j]):
        E6 = lagrange_lift(j,c)(T3)^6
        divisions[f'JU{i}'] = (E6*(T3-c), 1)
        divisions[f'JV{i}'] = (E6*(T29-2), 1)
        for record in selector_polynomials:
            if record['degree_residue_mod42'] != r or record['centre'] != c:
                continue
            arguments = (Q,G,v[f'JU{i}'] if record.get('raw_digit') == 'U' else v[f'JV{i}'])
            polynomial = record['polynomial']^3
            terminal = P.zero()
            for powers, coefficient in polynomial.dict().items():
                terminal += ZZ(coefficient)*prod(z^e for z,e in zip(arguments,powers))
            relations.append((record['label'], terminal, 1))
            inputs.append(E6)
    spec = relation_spec(relations, {'T3':3,'T29':29}, divisions)
    for record, initial in zip(spec['relations'][1:], inputs):
        record['input_polynomial'] = [
            [str(ZZ(c)), [int(e) for e in powers]]
            for powers,c in initial.dict().items()]
    return spec

groups = {}
for name,m,bound,threshold in [('G_mod49',2,350,56), ('Q_selectors_mod343',3,2450,392)]:
    specs = {r: make_specification(r,m) for r in range(0,42,2)}
    plan = json.loads((ROOT / f'relations/p7_mod49_{name}_native.json').read_text())
    assert {str(r):s for r,s in specs.items()} == plan['relation_specifications']
    groups[name] = {
        'exponent':m, 'ring':Integers(7^m), 'degrees':tuple(range(0,bound,2)),
        'threshold':threshold, 'specifications':specs,
        'sources':ROOT / 'source_data/p7_mod49_transfer_maps' / name,
    }
    print(name, '| modulus:', 7^m, '| degrees:', bound//2)


G_mod49 | modulus: 49 | degrees: 175
Q_selectors_mod343 | modulus: 343 | degrees: 1225


## 2. Recover intermediate elements and check the equations

For $7^a y=u$, the recorded data start with a coordinatewise preimage. Some files then add a specified element of the kernel of multiplication by $7^a$; others provide reusable matrices of division outputs. These are choices of intermediate elements, and every defining equation is still checked.

The archive identifier checks that the recorded elements correspond to the specified source, input rows, operators and relations. It is a file-consistency check, not a substitute for the mathematical checks.

At a recursive degree, the transferred lower sources and the selected complement must generate the whole source. Generation is tested modulo $7$, which suffices for these finite $7$-primary modules. Each lower source is verified first.


In [3]:
GENERATOR_BATCH_SIZE = 4

def compact_json(value):
    """
    Serialize the source data and presentation in the archive's JSON format.
    Return the text used to identify the corresponding verification data.
    Values must already be JSON-compatible, including Python integers.
    """
    return json.dumps(value, separators=(',', ':'), ensure_ascii=False)

def nim_sequence(values):
    """
    Encode a sequence of integers in the archive's '@[1, 2, ...]' format.
    This is used only for the identifier of the recorded data; no Nim
    computation is performed.
    """
    return '@[' + ', '.join(str(int(v)) for v in values) + ']'

def mixed_zero(value, moduli):
    """
    Check equality to zero in the source module in cyclic coordinates.
    
    The rows represent elements of M = direct_sum_j Z/moduli[j]Z.
    Each column is reduced modulo its own cyclic order, rather than the
    ambient modulus. Return True precisely when all rows represent zero.
    For the zero module, this condition holds vacuously.
    """
    return all(ZZ(value[i,j]) % order == 0 for i in range(value.nrows())
               for j, order in enumerate(moduli))

def canonical_preimage(rhs, divisor, moduli):
    """
    Recover the recorded y in the relation rhs = divisor*y on M.
    
    The rows of rhs are elements of M in cyclic coordinates. The divisor
    and cyclic orders are powers of the same prime. In each coordinate,
    take the least nonnegative residue of rhs and divide it by divisor,
    using the coordinatewise choice specified by the recorded data.
    
    Return the rows y after checking divisor*y = rhs in M. Failure of
    the required divisibility raises ArithmeticError. These are choices
    of intermediate elements, not a globally defined divided endomorphism.
    """
    output = zero_matrix(rhs.base_ring(), rhs.nrows(), rhs.ncols())
    for i in range(rhs.nrows()):
        for j, order in enumerate(moduli):
            entry = ZZ(rhs[i,j]) % order
            if entry % gcd(divisor, order):
                raise ArithmeticError(f'Division equation fails in coordinate ({i},{j})')
            output[i,j] = entry // divisor
    assert mixed_zero(divisor*output-rhs, moduli)
    return output


def polynomial_matrix(terms, variables, matrices, ring, rank):
    """
    Evaluate an ordinary polynomial in the supplied matrices.
    Apply the rightmost variable first, using the row-vector convention.
    A missing variable with positive exponent is rejected.
    """
    result = zero_matrix(ring, rank, rank)
    for coefficient,powers in terms:
        term = identity_matrix(ring,rank)
        for name,power in reversed(list(zip(variables,powers))):
            if power:
                term *= matrices[name]^power
        result += ZZ(coefficient)*term
    return result

@lru_cache(maxsize=int(12))
def load_source(group_name, d, q):
    """
    Load the signed source, its ordinary Hecke actions and transfer maps.
    Apply the orientation exactly once. Check that each action respects
    the cyclic orders. The presentation and Hecke descent are trusted inputs.
    Only a small number of sources are retained in memory.
    """
    group = groups[group_name]
    ring, m = group['ring'], group['exponent']
    path = group['sources'] / f'degree_{d}.npz'
    prefix = f'q{q%2}_'
    with np.load(path, allow_pickle=False) as archive:
        metadata = json.loads(bytes(archive['metadata_json']).decode())
        assert metadata['prime'] == 7 and metadata['exponent'] == m and metadata['degree'] == d
        orders = [int(7^int(e)) for e in archive[prefix+'order_exponents']]
        rank = len(orders)
        matrices = {}
        for name,index in [('T3',3),('T29',29)]:
            raw = archive[prefix+name]
            assert raw.shape == (rank,rank)
            action = matrix(ring,rank,rank,[int(x) for x in raw.flat])
            action *= ring(power_mod(index,q*7^(m-1),7^m))
            assert all(orders[i]*ZZ(action[i,j]) % orders[j] == 0
                       for i in range(rank) for j in range(rank))
            matrices[name] = action
        maps = {}
        recursive = bool(archive[prefix+'recursive'][0])
        if recursive:
            for name in ('transfer_a','transfer_b','complement_images'):
                raw = archive[prefix+name]
                maps[name] = matrix(ring,raw.shape[0],raw.shape[1],
                                    [int(x) for x in raw.flat])
    return orders, matrices, maps, recursive

def select_complement(inherited, complement, rank, ring):
    """
    Select complement rows completing the transferred rows to generators.
    Perform elimination over F_7 in archive order. Full rank proves generation
    of the finite source by Nakayama's lemma. Return the selected original rows.
    """
    pivots = {}
    selected = []
    for block,rows in enumerate((inherited,complement)):
        for index,row in enumerate(rows.rows()):
            v = vector(GF(7),row)
            for j in range(rank):
                if not v[j]:
                    continue
                if j in pivots:
                    v -= v[j]*pivots[j]
                else:
                    v /= v[j]
                    pivots[j] = v
                    if block == 1:
                        selected.append(index)
                    break
            if len(pivots) == rank:
                return complement.matrix_from_rows(selected)
    if rank == 0:
        return zero_matrix(ring,0,0)
    raise ArithmeticError('Transferred rows and complement do not generate the source.')

def packet_binding(spec, metadata, orders, operators, m, inputs):
    """
    Identify the recorded intermediate elements for these exact inputs.
    Include the tested complement rows as well as the source and relations.
    The identifier prevents mismatched files; replay checks the equations.
    """
    parts = [compact_json(spec), f'7:{m}', compact_json(metadata),
             nim_sequence(orders),
             f'{inputs.nrows()}:{inputs.ncols()}:'+nim_sequence(inputs.list())]
    for name in spec['variables']:
        if name in operators:
            parts.append(name+':'+nim_sequence(operators[name].list()))
    return hashlib.sha1(''.join(parts).encode()).hexdigest().upper()

def load_record(directory, binding, relation, metadata, count, modulus):
    """
    Read the recorded choices for a single presented relation.
    Check their identifier, source, input count and working modulus.
    These checks do not replace the defining division and terminal equations.
    """
    suffix = hashlib.sha1(relation['name'].encode()).hexdigest().upper()
    path = directory / f'{binding}_{suffix}.json.gz'
    with gzip.open(path,'rt') as stream:
        record = json.load(stream)
    assert record['schema'] in ('hecke.compact-relation-witness.v1','hecke.compact-relation-witness.v2')
    assert record['binding'] == binding and record['relation'] == relation['name']
    assert record['source'] == metadata and record['input_count'] == count
    assert record['working_modulus'] == modulus
    assert record['independent'] is False
    return record

def defining_equations(spec, relation):
    """
    Expand the defining equations using the recorded common-chain order.
    Number the intermediate elements starting with input zero, sharing
    repeated applications to the same input. Append terminal membership
    in 7^b M. No simplification of multivalued relations is performed.
    """
    variables = spec['variables']
    assert not spec.get('presentations') and 'input_presentation' not in relation
    definitions = {item['variable']:item for item in spec['divisions']}
    steps, cache = [], {}
    def polynomial(terms, input_node):
        """Compile a polynomial, combining identical terms in archive order."""
        collected = {}
        for coefficient,powers in terms:
            key = tuple(powers)
            collected[key] = (collected.get(key,0)+ZZ(coefficient)) % spec['coefficient_modulus']
        outputs = {}
        for powers,coefficient in collected.items():
            if not coefficient:
                continue
            node = input_node
            for name,power in reversed(list(zip(variables,powers))):
                for _ in range(power):
                    node = apply_relation(name,node)
            outputs[node] = (outputs.get(node,0)+coefficient) % spec['coefficient_modulus']
        return [(node,c) for node,c in outputs.items() if c]
    def apply_relation(name, input_node):
        """Record one operator or division application and return its output number."""
        key = (name,input_node)
        if key in cache:
            return cache[key]
        if name in spec['hecke_operators']:
            step = (name,input_node,0,None)
        else:
            definition = definitions[name]
            terms = polynomial(definition['numerator'],input_node)
            step = (name,input_node,7^definition['power'],terms)
        steps.append(step)
        cache[key] = len(steps)
        return len(steps)
    terms = polynomial(relation['polynomial'],0)
    steps.append((None,0,7^relation['terminal_power'],terms))
    return steps

def restore_outputs(spec, record, operators, orders, ring):
    """
    Decode reusable matrices of division outputs when the file provides them.
    Verify their cyclic well-definedness and their numerator equations.
    Return an empty dictionary for coordinatewise choices instead.
    """
    if 'recipe' not in record:
        return {}
    recipe = record['recipe']
    assert recipe['schema'] == 'hecke.global-division-recipe.v1'
    saved = iter(recipe['outputs'])
    matrices = dict(operators)
    definitions = {item['variable']:item for item in spec['divisions']}
    rank = len(orders)
    for index,name in enumerate(spec['variables']):
        if name in operators:
            continue
        definition = definitions[name]
        numerator = polynomial_matrix(definition['numerator'],spec['variables'],matrices,ring,rank)
        item = next(saved)
        assert item['variable'] == index and len(item['entries']) == rank*rank
        assert all(0 <= entry < orders[k%rank] for k,entry in enumerate(item['entries']))
        chosen = matrix(ring,rank,rank,item['entries'])
        assert mixed_zero(7^definition['power']*chosen-numerator,orders)
        assert all(orders[i]*ZZ(chosen[i,j]) % orders[j] == 0
                   for i in range(rank) for j in range(rank))
        matrices[name] = chosen
    assert next(saved,None) is None
    return matrices

def replay_equations(spec, relation, record, inputs, operators, orders, ring):
    """
    Recover and check the intermediate elements on the prescribed input rows.
    Apply an ordinary input selector first. Reconstruct each recorded kernel
    correction or reusable division output, checking every division equation
    and the final membership in 7^b M. No equation is solved by search.
    """
    steps = defining_equations(spec,relation)
    reusable = restore_outputs(spec,record,operators,orders,ring)
    corrections = {}
    previous = None
    for node,row,col,coefficient in record['choices']:
        key = (node,row,col)
        assert previous is None or previous < key
        previous = key
        assert 1 <= node < len(steps) and 0 <= row < inputs.nrows() and 0 <= col < len(orders)
        divisor = steps[node-1][2]
        assert divisor > 1 and 0 < coefficient < gcd(divisor,orders[col])
        corrections[key] = coefficient
    assert not (reusable and corrections)
    initial = inputs
    if 'input_polynomial' in relation:
        initial = inputs*polynomial_matrix(relation['input_polynomial'],spec['variables'],
                                           operators,ring,len(orders))
    uses = [0]*(len(steps)+1)
    for name,input_node,divisor,terms in steps:
        dependencies = [input_node] if divisor == 0 else [node for node,c in terms]
        if reusable and divisor and name is not None:
            dependencies = dependencies+[input_node]
        for node in dependencies:
            uses[node] += 1
    for start in range(0,initial.nrows(),GENERATOR_BATCH_SIZE):
        count = min(GENERATOR_BATCH_SIZE,initial.nrows()-start)
        values = {0:initial.matrix_from_rows(range(start,start+count))}
        remaining = list(uses)
        for number,(name,input_node,divisor,terms) in enumerate(steps,start=1):
            if divisor == 0:
                output = values[input_node]*operators[name]
                dependencies = [input_node]
            else:
                rhs = zero_matrix(ring,count,len(orders))
                for node,c in terms:
                    rhs += c*values[node]
                output = canonical_preimage(rhs,divisor,orders)
                if reusable and name is not None:
                    output = values[input_node]*reusable[name]
                else:
                    for i in range(count):
                        for j,order in enumerate(orders):
                            coefficient = corrections.get((number,start+i,j),0)
                            output[i,j] += (order//gcd(divisor,order))*coefficient
                assert mixed_zero(divisor*output-rhs,orders), (relation['name'],number)
                dependencies = [node for node,c in terms]
                if reusable and name is not None:
                    dependencies = dependencies+[input_node]
            values[number] = output
            for node in dependencies:
                remaining[node] -= 1
                if remaining[node] == 0:
                    del values[node]
            if remaining[number] == 0:
                del values[number]

verified_cases = {}

def replay_mod49_case(group_name, d, q):
    """
    Verify the whole finite source using its recorded recursive presentation.
    First verify each required lower source, then the transfer homomorphisms
    and Hecke compatibility. Replay the selected complement and check that
    it completes the inherited rows to generators. Cache successful results
    only for this notebook session; missing or invalid data stop verification.
    """
    key = (group_name,int(d),int(q))
    if key in verified_cases:
        return verified_cases[key]
    group = groups[group_name]
    assert d in group['degrees'] and q in range(6)
    ring,m = group['ring'],group['exponent']
    orders,operators,maps,recursive = load_source(group_name,int(d),int(q))
    rank = len(orders)
    spec = group['specifications'][(d+14*q)%42]
    inherited = zero_matrix(ring,0,rank)
    if recursive:
        for label,shift,lower_q in [('a',7^m*6,q),('b',7^(m-1)*8,(q+1)%6)]:
            lower_d = d-shift
            if lower_d < 0:
                continue
            lower_spec = group['specifications'][(lower_d+14*lower_q)%42]
            assert lower_spec == spec
            replay_mod49_case(group_name,lower_d,lower_q)
            lower_orders,lower_operators,_,_ = load_source(group_name,int(lower_d),int(lower_q))
            transfer = maps['transfer_'+label]
            assert transfer.dimensions() == (len(lower_orders),rank)
            assert all(lower_orders[i]*ZZ(transfer[i,j]) % orders[j] == 0
                       for i in range(len(lower_orders)) for j in range(rank))
            for name in operators:
                assert mixed_zero(lower_operators[name]*transfer-transfer*operators[name],orders)
            inherited = inherited.stack(transfer)
        inputs = select_complement(inherited,maps['complement_images'],rank,ring)
    else:
        inputs = identity_matrix(ring,rank)
    metadata = {'degree':int(d),'orientation':int(q),'sign':int((-1)^q),
                'source_scope':'manin','construction':'recursive',
                'ideal':None,'ideal_variable_hecke_indices':[int(3),int(29)]}
    binding = packet_binding(spec,metadata,orders,operators,m,inputs)
    for relation in spec['relations']:
        record = load_record(VERIFICATION_DIRECTORY/group_name/'packets',binding,
                             relation,metadata,inputs.nrows(),7^m)
        replay_equations(spec,relation,record,inputs,operators,orders,ring)
    result = {'group':group_name,'degree':d,'orientation':q,'sign':(-1)^q,
              'rank':rank,'complement_inputs':inputs.nrows(),'recursive':recursive,
              'packets_loaded':len(spec['relations']),'passed':True}
    verified_cases[key] = result
    return result


## The G relation, at precision 49

Verify in ascending degree order. Below the induction base, the requested cases have $q=0$; the recursive checks also verify any lower orientations they need. For a short trial, replace the degree list by a few degrees. A successful finite loop is not by itself a proof of all-weight propagation.


In [4]:
results_G_mod49 = []
for d in groups['G_mod49']['degrees']:
    orientations = range(6) if d >= groups['G_mod49']['threshold'] else (0,)
    for q in orientations:
        test = replay_mod49_case('G_mod49',d,q)
        results_G_mod49.append(test)
        print(test,flush=True)
print('Cases replayed:',len(results_G_mod49))
print('ALL REQUESTED G_mod49 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


{'group': 'G_mod49', 'degree': 0, 'orientation': 0, 'sign': 1, 'rank': 0, 'complement_inputs': 0, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 2, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 4, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 6, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 8, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 10, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2, 'recursive': False, 'packets_loaded': 1, 'passed': True}
{'group': 'G_mod49', 'degree': 12, 'orientation': 0, 'sign': 1, 'rank

## The Q and selector relations, at precision 343

Verify in ascending degree order. Below the induction base, the requested cases have $q=0$; the recursive checks also verify any lower orientations they need. For a short trial, replace the degree list by a few degrees. A successful finite loop is not by itself a proof of all-weight propagation.


In [5]:
results_Q_selectors_mod343 = []
for d in groups['Q_selectors_mod343']['degrees']:
    orientations = range(6) if d >= groups['Q_selectors_mod343']['threshold'] else (0,)
    for q in orientations:
        test = replay_mod49_case('Q_selectors_mod343',d,q)
        results_Q_selectors_mod343.append(test)
        print(test,flush=True)
print('Cases replayed:',len(results_Q_selectors_mod343))
print('ALL REQUESTED Q_selectors_mod343 FINITE-SOURCE CONDITIONS VERIFIED IN SAGE')


{'group': 'Q_selectors_mod343', 'degree': 0, 'orientation': 0, 'sign': 1, 'rank': 0, 'complement_inputs': 0, 'recursive': False, 'packets_loaded': 11, 'passed': True}
{'group': 'Q_selectors_mod343', 'degree': 2, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 11, 'passed': True}
{'group': 'Q_selectors_mod343', 'degree': 4, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 11, 'passed': True}
{'group': 'Q_selectors_mod343', 'degree': 6, 'orientation': 0, 'sign': 1, 'rank': 1, 'complement_inputs': 1, 'recursive': False, 'packets_loaded': 11, 'passed': True}
{'group': 'Q_selectors_mod343', 'degree': 8, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2, 'recursive': False, 'packets_loaded': 11, 'passed': True}
{'group': 'Q_selectors_mod343', 'degree': 10, 'orientation': 0, 'sign': 1, 'rank': 2, 'complement_inputs': 2, 'recursive': False, 'packets_loaded': 11, 'passed': Tru

KeyboardInterrupt: 

## Optional: replay the same recorded data with Nim

This is an alternative to the Sage loops. Run the imports and relation definitions first. Set `RUN_NIM_REPLAY=True` to enable it and adjust `NIM_WORKERS` (default 4).

Each case checks the recursive lower sources as well as its complement. Fresh processes do not share the Sage cache and may repeat lower-degree work. Only saved intermediate elements are replayed; no new choices are searched for. After a failed batch, no further batch is started.


In [7]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

RUN_NIM_REPLAY = True
NIM_WORKERS = 4
VERIFIER = ROOT / 'nim/.verify-hecke-relations-build/verify_hecke_relations'

def replay_nim_case(case):
    """
    Replay one case and its recursive dependencies with the native verifier.
    Load archived sources and recorded intermediate elements in strict replay
    mode. Return the report; missing data or failed equations raise an error.
    """
    name,d,q = case
    group = groups[name]
    request = {
        'compute': {
            'prime':int(7),'exponent':int(group['exponent']),
            'degree':int(d),'orientation':int(q),
            'recursive':True,'recursive_verification':True,
            'archive_directory':str(group['sources']),
        },
        'relations':group['specifications'][(d+14*q)%42],
        'witness_directory':str(VERIFICATION_DIRECTORY/name/'packets'),
        'witness_mode':'replay',
    }
    if not VERIFIER.is_file():
        raise FileNotFoundError(VERIFIER)
    environment = os.environ.copy()
    environment['LD_LIBRARY_PATH'] = (str(Path(sys.prefix)/'lib')+os.pathsep+
                                      environment.get('LD_LIBRARY_PATH',''))
    process = subprocess.run([str(VERIFIER),'-'],input=json.dumps(request),
                             text=True,capture_output=True,env=environment)
    if not process.stdout.strip():
        raise RuntimeError(process.stderr)
    report = json.loads(process.stdout)
    if process.returncode or not report.get('passed',False):
        raise RuntimeError(f'{case}: {report}')
    assert report['verification_scope'] == 'whole_source'
    assert report['recursive_verification'] is True
    assert report['archived_actions_reused'] is True
    assert len(report['relations']) == len(request['relations']['relations'])
    for result in report['relations']:
        assert result['passed'] is True and result['witness_file_reused'] is True
    for item in report['recursive_trace']:
        assert item['state'] == 'passed'
    return {'group':name,'degree':d,'orientation':q,'passed':True}

def run_parallel_nim_replay(cases, workers=4):
    """
    Replay cases in bounded parallel batches with an adjustable worker count.
    A fresh pool for each batch releases worker resources afterwards.
    Stop after a failed batch without scheduling new cases; return successful
    reports only if every requested case has passed.
    """
    if workers != int(workers) or workers < 1:
        raise ValueError('workers must be positive')
    workers = int(workers)
    results = []
    for start in range(0,len(cases),workers):
        batch = cases[start:start+workers]
        failures = []
        with ProcessPoolExecutor(max_workers=len(batch),
                                 mp_context=get_context('fork')) as executor:
            futures = {executor.submit(replay_nim_case,case):case for case in batch}
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    print(result,flush=True)
                except Exception as error:
                    failures.append((futures[future],str(error)))
        if failures:
            raise RuntimeError(f'Batch failed; no further cases scheduled: {failures}')
    return results

if RUN_NIM_REPLAY:
    nim_cases = [
        (name,int(d),int(q))
        for name,group in groups.items()
        for d in group['degrees']
        for q in (range(6) if d >= group['threshold'] else (0,))
    ]
    nim_results = run_parallel_nim_replay(nim_cases,NIM_WORKERS)
    print('ALL REQUESTED FINITE-SOURCE CONDITIONS VERIFIED BY NIM REPLAY')
else:
    print('Optional Nim replay is disabled. Set RUN_NIM_REPLAY = True to run it.')


{'group': 'G_mod49', 'degree': 0, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 6, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 2, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 4, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 8, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 12, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 10, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 14, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 16, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 20, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 22, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 18, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 24, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 26, 'orientation': 0, 'passed': True}
{'group': 'G_mod49', 'degree': 28, 'ori

## Strong realization of the permitted signatures

Check that every permitted signature has a saved strong representative.
Write $a_3=c+7x$ and $a_{29}=2+7y$, with $x,y\in\mathbf F_7$.
Evaluate the polynomials $F_j$, $R_{r,c}^{(\nu)}$, $\phi_{r,c}$
and $\psi_{r,c}$ already defined above, with $r=k-2\bmod42$.
They give 15 pairs in each even **weight** residue $k\bmod42$,
hence 315 signatures in total.

Compare this set with the completed strong-eigenform scan in
`strong_signatures/p7_m2/`, through weight 380. The code checks
the scan parameters and coverage and compares its saved representatives;
it does not recompute eigenforms or repeat their number-field reductions.
Those computations are inputs from the strong-signature producer.

Run the imports and relation definitions first. No source replay loop
is needed again. This finite realization check supplies the converse;
the all-weight assertion also uses propagation and completed-Hecke
generation from the manuscript. A bounded scan alone does not prove it.


In [ ]:
STRONG_SIGNATURE_DIRECTORY_MOD49 = ROOT / 'strong_signatures/p7_m2'
STRONG_WEIGHT_BOUND_MOD49 = 380
STRONG_WEIGHT_PERIOD_MOD49 = 42


def permitted_mod49_signatures():
    """
    Evaluate the displayed scalar relations on all proposed digits.
    Use Q=F_j(a_3)/49 and G=(a_29-2-a_3(a_3-alpha)(a_3-beta))/7.
    Their residues and the raw digits x,y must satisfy every
    coordinate and graph polynomial in the selected branch.
    Return distinct pairs indexed by classical weight residue.
    """
    field = GF(7)
    permitted = {}
    for k_residue in range(0, 42, 2):
        r = (k_residue - 2) % 42
        j = r % 6
        alpha, beta = centres[j][1:]
        pairs = set()
        for c in centres[j]:
            branch = selector_data['by_branch'][r, c]
            for x in range(7):
                for y in range(7):
                    a3, a29 = c + 7*x, 2 + 7*y
                    numerator_Q = F_j[j](a3)
                    numerator_G = a29 - 2 - a3*(a3-alpha)*(a3-beta)
                    assert numerator_Q % 49 == 0 and numerator_G % 7 == 0
                    Q_value, G_value = field(numerator_Q // 49), field(numerator_G // 7)
                    passed = True
                    for record in branch:
                        polynomial = record['polynomial']
                        if record['family'] == 'compact_W_coordinate':
                            value = polynomial(Q_value, G_value)
                        else:
                            digit = x if record['raw_digit'] == 'U' else y
                            value = polynomial(Q_value, G_value, field(digit))
                        if value != 0:
                            passed = False
                            break
                    if passed:
                        pairs.add((ZZ(a3), ZZ(a29)))
        assert len(pairs) == 15
        permitted[k_residue] = pairs
    assert sum(map(len, permitted.values())) == 315
    return permitted


strong_mod49_expected = permitted_mod49_signatures()


In [ ]:
def load_mod49_strong_signatures(directory, bound):
    """
    Read the representatives in the completed modulo-49 scan summary.
    Check the prime, precision, selected coordinates and weight coverage.
    This checks the saved summary, not the producer's eigenform or
    prime-ideal calculations. Do not silently omit nonrational packets.
    """
    directory = Path(directory)
    status = json.loads((directory / 'status.json').read_text())
    summary = json.loads((directory / 'summary.json').read_text())
    assert status['schema'] == 'hecke.strong-signature-status.v1'
    assert status['state'] == 'completed' and status['failed'] == []
    assert summary['schema'] == 'hecke.strong-signature-summary.v1'
    for record in (status, summary):
        assert record['prime'] == 7 and record['exponent'] == 2
        assert record['maximum_weight'] == bound and record['bounded_scan_complete']
    assert summary['hecke_indices'] == [3, 29]
    assert summary['completed_weights'] == list(range(2, bound + 1, 2))
    assert summary['nonrational_packet_count'] == 0
    assert summary['rational_signature_count'] == len(summary['rational_signatures'])
    signatures = []
    for entry in summary['rational_signatures']:
        weight = ZZ(entry['weight'])
        pair = tuple(ZZ(value) for value in entry['eigenvalue_residues'])
        assert 2 <= weight <= bound and weight % 2 == 0
        assert entry['weight_residue'] == weight % 42
        assert len(pair) == 2 and all(0 <= value < 49 for value in pair)
        signatures.append((weight, *pair))
    return signatures


strong_mod49_signatures = load_mod49_strong_signatures(
    STRONG_SIGNATURE_DIRECTORY_MOD49, STRONG_WEIGHT_BOUND_MOD49)
strong_mod49_realized = {}
for weight, a3, a29 in strong_mod49_signatures:
    k_residue = weight % STRONG_WEIGHT_PERIOD_MOD49
    strong_mod49_realized.setdefault(k_residue, set()).add((a3, a29))
print('Saved strong representatives:', len(strong_mod49_signatures))
print('Largest representative weight:', max(weight for weight, _, _ in strong_mod49_signatures))


In [ ]:
print('k mod 42 | (a_3, a_29) mod 49')
print('-' * 100)
for k_residue in sorted(strong_mod49_expected):
    pairs = ', '.join(
        f'({a1}, {a2})'
        for a1, a2 in sorted(strong_mod49_realized.get(k_residue, set()))
    )
    print(f'{k_residue:3d} | {pairs}')

assert strong_mod49_realized == strong_mod49_expected
print('Every permitted modulo-49 signature has a strong representative.')
